In [40]:
import torch
import torch.nn as nn
import torch.optim as optim
from torchvision import datasets, transforms
from torch.utils.data import DataLoader

In [41]:
#pytorch expects channel first(C, H, W) converts PIL image(0-255 int) to float(0-1) pytorch tensor. hence ToTensor()
transform = transforms.Compose([
    transforms.Resize((128, 128)),
    transforms.ToTensor(),
    transforms.Normalize((0.5,), (0.5,))
])

In [42]:
train_dataset = datasets.ImageFolder(
    root = 'data/Training',
    transform=transform
)

test_dataset = datasets.ImageFolder(
    root = 'data/Testing',
    transform=transform
)

In [43]:
train_dataset.class_to_idx

{'glioma': 0, 'meningioma': 1, 'notumor': 2, 'pituitary': 3}

In [44]:
train_loader = DataLoader(
    train_dataset,
    batch_size=32,
    shuffle=True
)
test_loader = DataLoader(
    test_dataset,
    batch_size=32,
    shuffle=True
)

In [45]:
#quick check
images, labels = next(iter(train_loader))
print(images.shape)
print(labels.shape)

torch.Size([32, 3, 128, 128])
torch.Size([32])


In [46]:
class BrainTumorCNN(nn.Module):
    def __init__(self):
        super().__init__().__init__()

        self.conv1 = nn.Conv2d(
            in_channels=3,
            out_channels=16,
            kernel_size=3,
            padding=1
        )

        self.conv2 = nn.Conv2d(
            in_channels=16,
            out_channels=32,
            kernel_size=3,
            padding =1
        )

        self.pool = nn.MaxPool2d(kernel_size=2, stride = 2)
        self.relu = nn.ReLU()

        self.fc1= nn.Linear(32*32*32, 128) #fc = fully connected layer, 32*32*32 = 32768 inputs and 128 outputs
        self.fc2 = nn.Linear(128, 4) #128 inputs and 4 outputs

    def forward(self, x):
        x = self.pool(self.relu(self.conv1(x))) #first conv pass
        x = self.pool(self.relu(self.conv2(x))) #second conv pass

        x = torch.flatten(x, 1) 

        x=self.relu(self.fc1(x))
        x= self.fc2(x) #raw class scores

        return x


In [47]:
model = BrainTumorCNN()
print(model)

BrainTumorCNN(
  (conv1): Conv2d(3, 16, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
  (conv2): Conv2d(16, 32, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
  (pool): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
  (relu): ReLU()
  (fc1): Linear(in_features=32768, out_features=128, bias=True)
  (fc2): Linear(in_features=128, out_features=4, bias=True)
)


In [48]:
criteria = nn.CrossEntropyLoss()#standard fro multiclass
optimizer = torch.optim.Adam(model.parameters(), lr=0.001)

In [49]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = model.to(device)

In [50]:
epochs = 10

for epoch in range(epochs):
    current_loss = 0

    for inputs, labels in train_loader:
        inputs, labels = inputs.to(device), labels.to(device)

        optimizer.zero_grad() #clear old grads

        outputs = model(inputs) #forward pass

        loss =  criteria(outputs, labels)
        loss.backward() #backward prop, on;y app;y chain rule to check how much each parameter/weight or layer is responsible for an error. does not change weights, only calculates

        optimizer.step() #update weights according to what the backward calculated

        current_loss += loss.item()

    print(f"Epoch {epoch+1}/{epochs}, Loss: {current_loss/len(train_loader): .4f}")


Epoch 1/10, Loss:  0.6083
Epoch 2/10, Loss:  0.3112
Epoch 3/10, Loss:  0.1859
Epoch 4/10, Loss:  0.0935
Epoch 5/10, Loss:  0.0629
Epoch 6/10, Loss:  0.0345
Epoch 7/10, Loss:  0.0189
Epoch 8/10, Loss:  0.0095
Epoch 9/10, Loss:  0.0047
Epoch 10/10, Loss:  0.0014


In [51]:
#Validation check
correct = 0
total = 0

model.eval() #eval mode

with torch.no_grad():
    for inputs, labels in test_loader:
        inputs, labels = inputs.to(device), labels.to(device)

    outputs = model(inputs)
    _, predicted = torch.max(outputs, 1) #get classs with highest score
    total+= labels.size(0)
    correct+=(predicted==labels).sum().item()

print(f"Validation Accuracy: {100* correct/total:.2f}%")

Validation Accuracy: 87.50%
